# 01 · Preprocessing

Inspect → harmonize → clean → optionally reserve an unseen attack → split → compress extreme SECOND_BYTES values → fit on train → transform/write each split in batches → release that split.

One dataset per run. Initial loading, splitting, and fitting still need RAM. Saved Parquet files contain ready-to-use transformed features.

## 0. Setup and Imports

In [ ]:
PLATFORM = 'kaggle'  # 'colab' or 'kaggle'
assert PLATFORM in ('colab', 'kaggle')

In [ ]:
if PLATFORM == 'colab':
    %load_ext cudf.pandas
    %load_ext cuml.accel

    from google.colab import drive
    drive.mount('/content/drive')
    INPATH = '/content/drive/MyDrive/Masters/Datasets/NF-UQ-NIDS(Splitted_Dataset)'
    OUTPATH = '/content/drive/MyDrive/Masters/IDS_repo/My repo'

else :
    INPATH = '/kaggle/input/datasets/omarahmed001/ids-datasets'
    OUTPATH = '/kaggle/working/project'


In [ ]:
%pip install -q "scikit-learn>=1.6" category_encoders
from pathlib import Path
from fastai.tabular.all import df_shrink
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.base import clone
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler, QuantileTransformer
from sklearn.preprocessing import FunctionTransformer, OrdinalEncoder
from sklearn.decomposition import PCA
from category_encoders import BinaryEncoder

# Fixed transform: no fitted limits and no row removal.
EXTREME_COLUMNS = ['SRC_TO_DST_SECOND_BYTES', 'DST_TO_SRC_SECOND_BYTES']

def compress_extreme_values(X):
    """Compress nonnegative SECOND_BYTES measurements before imputation."""
    X = X.copy()
    for col in EXTREME_COLUMNS:
        if col in X.columns:
            values = X[col].astype('float64').replace([np.inf, -np.inf], np.nan)
            if values.lt(0).any():
                raise ValueError(f'{col} contains negative byte measurements.')
            X[col] = np.log1p(values)
    return X


In [ ]:
DATA_DIR = Path(INPATH)
FILES = {
    'NF_UNSW_NB15': 'NF-UNSW-NB15-v2.parquet',
    'NF_BOT_IOT': 'NF-BoT-IoT-v2.parquet',
    'NF_CSE_CIC_IDS2018': 'NF-CSE-CIC-IDS2018-v2.parquet',
    'NF_TON_IOT': 'NF-ToN-IoT-v2.parquet',
    'CIC-BoT-IoT': 'CIC-BoT-IoT-V2.parquet',
    'CSE-CIC-IDS2018': 'CSE-CIC-IDS2018.parquet',
    'UNSW-NB15': 'UNSW-NB15.parquet'
}

In [ ]:
DATASET = 'NF_TON_IOT'  # Change this to run one of the other datasets.
HOLD_OUT_ATTACK = False  # False: include every attack class in the 70/15/15 split.
HELD_OUT_ATTACK = 'Worms'  # Used only when HOLD_OUT_ATTACK is True.
SEED = 42

## 1. Load and inspect

In [ ]:
df = pd.read_parquet(DATA_DIR / FILES[DATASET])
print(DATASET, df.shape)
display(df.head())
display(df['Attack'].value_counts())

In [ ]:
df.Label.value_counts()

In [ ]:
display(pd.DataFrame({
    'Dtype': df.dtypes,
    'Unique Values': df.nunique()
}))

### Actual sample versus the full original dataset

Keep up to `MAX_PER_CLASS = 300_000` randomly selected rows per `Attack`; retain every row from classes smaller than the cap. The cells compare this actual sample with the full original data before encoding, cleaning, splitting, or fitted transformations. No second reference sample or 50,000-row diagnostic cap is used.

The class table measures the deliberate change in class proportions. Numerical features are compared within each Attack using KS distance: 0 means matching empirical CDFs; 0.02 means a maximum CDF gap of 2 percentage points. Nominal fields use total variation distance between category frequencies (0 means matching frequencies). Missing and infinite rates are reported separately. Both distances range from 0 to 1; no universal pass/fail cutoff is imposed. Read the complete tables in `numeric_distribution_report` and `categorical_distribution_report`.

These are descriptive comparisons. KS p-values are not reported because the sample is a subset of the original, many features are discrete, and a small p-value would not establish a practically important difference. Marginal agreement does not prove matching correlations, rare subgroups, or tails. Keeping every minority row gives matching raw distributions for that class; it does not add new independent observations.

`CHECK_SAMPLE_DISTRIBUTION = False` skips the feature comparisons while retaining the class table and releasing the original. Full-data comparisons may take time; they process one column at a time and print each class. Rerun **Load and inspect** before repeating sampling.

Sampling remains before the split in this workflow: validation/test therefore reflect the capped class mix. For evaluation at the original class proportions, split the original data first and cap training only. Preprocessing is still fitted after splitting.

In [ ]:
import gc

MAX_PER_CLASS = 300_000
CHECK_SAMPLE_DISTRIBUTION = True

# Retain the full original data until the comparison below is finished.
# This assignment does not copy the original DataFrame.
original_df = df
original_counts = original_df['Attack'].value_counts(dropna=False)

sampled_train = pd.concat(
    [
        group.sample(n=min(len(group), MAX_PER_CLASS), random_state=SEED)
        for _, group in original_df.groupby(
            'Attack', observed=True, dropna=False
        )
    ],
    ignore_index=True,
)
df = sampled_train.sample(frac=1, random_state=SEED).reset_index(drop=True)
del sampled_train

sample_counts = df['Attack'].value_counts(dropna=False)
class_comparison = pd.DataFrame({
    'Original rows': original_counts,
    'Sample rows': sample_counts,
}).fillna(0).astype('int64').rename_axis('Attack')
class_comparison['Original (%)'] = (
    100 * class_comparison['Original rows'] / len(original_df)
)
class_comparison['Sample (%)'] = (
    100 * class_comparison['Sample rows'] / len(df)
)
class_comparison['Difference (percentage points)'] = (
    class_comparison['Sample (%)'] - class_comparison['Original (%)']
)
class_comparison['Rows retained (%)'] = (
    100 * class_comparison['Sample rows'] / class_comparison['Original rows']
)
print(f'Original: {len(original_df):,} rows | Actual sample: {len(df):,} rows')
display(class_comparison.round(4))
class_tv_distance = (
    class_comparison['Sample (%)'] - class_comparison['Original (%)']
).abs().sum() / 200
print(f'Class-proportion TV distance: {class_tv_distance:.6f}')

In [ ]:
from scipy.stats import ks_2samp

# Nominal fields are compared by category frequencies, not numeric ordering.
# Add another dataset-specific nominal column here if needed.
sampling_nominal_columns = ['PROTOCOL', 'L7_PROTO', 'DNS_QUERY_TYPE', 'FTP_COMMAND_RET_CODE',
                            'ICMP_IPV4_TYPE', 'ICMP_TYPE', 'L4_SRC_PORT', 'L4_DST_PORT', 'TCP_FLAGS',
                            'CLIENT_TCP_FLAGS', 'SERVER_TCP_FLAGS', 'IPV4_SRC_ADDR', 'IPV4_DST_ADDR', 'DNS_QUERY_ID']

if CHECK_SAMPLE_DISTRIBUTION:
    categorical_features = [
        c for c in original_df.columns
        if c not in ('Label', 'Attack')
        and (c in sampling_nominal_columns
             or not pd.api.types.is_numeric_dtype(original_df[c]))
    ]
    numeric_features = [
        c for c in original_df.select_dtypes(include='number').columns
        if c not in ('Label', 'Attack') and c not in categorical_features
    ]

    numeric_results = []
    categorical_results = []

    for attack in original_counts.index:
        print(f'Comparing full original vs actual sample: {attack}', flush=True)
        original_mask = (original_df['Attack'].isna() if pd.isna(attack)
                         else original_df['Attack'].eq(attack))
        sample_mask = (df['Attack'].isna() if pd.isna(attack)
                       else df['Attack'].eq(attack))

        # Use every row in both datasets; process one feature at a time.
        for col in numeric_features:
            a = original_df.loc[original_mask, col].to_numpy(
                dtype=np.float64, na_value=np.nan
            )
            b = df.loc[sample_mask, col].to_numpy(
                dtype=np.float64, na_value=np.nan
            )
            finite_a = a[np.isfinite(a)]
            finite_b = b[np.isfinite(b)]

            distance = (
                ks_2samp(finite_a, finite_b, method='asymp').statistic
                if len(finite_a) and len(finite_b) else np.nan
            )
            numeric_results.append({
                'Attack': attack,
                'feature': col,
                'Original rows': len(a),
                'Sample rows': len(b),
                'KS_D': distance,
                'Original missing (%)': 100 * np.isnan(a).mean(),
                'Sample missing (%)': 100 * np.isnan(b).mean(),
                'Original infinite (%)': 100 * np.isinf(a).mean(),
                'Sample infinite (%)': 100 * np.isinf(b).mean(),
            })
            del a, b, finite_a, finite_b

        for col in categorical_features:
            original_frequencies = original_df.loc[
                original_mask, col
            ].value_counts(normalize=True, dropna=False)
            sample_frequencies = df.loc[
                sample_mask, col
            ].value_counts(normalize=True, dropna=False)
            frequencies = pd.concat([
                original_frequencies.rename('original'),
                sample_frequencies.rename('sample'),
            ], axis=1).fillna(0)
            tv_distance = (
                frequencies['original'] - frequencies['sample']
            ).abs().sum() / 2
            categorical_results.append({
                'Attack': attack,
                'feature': col,
                'TV_distance': tv_distance,
            })
            del original_frequencies, sample_frequencies, frequencies

        del original_mask, sample_mask

    numeric_distribution_report = pd.DataFrame(numeric_results, columns=[
        'Attack', 'feature', 'Original rows', 'Sample rows', 'KS_D',
        'Original missing (%)', 'Sample missing (%)',
        'Original infinite (%)', 'Sample infinite (%)',
    ]).sort_values('KS_D', ascending=False)
    categorical_distribution_report = pd.DataFrame(
        categorical_results, columns=['Attack', 'feature', 'TV_distance']
    ).sort_values('TV_distance', ascending=False)

    print('Largest numerical distribution differences (KS distance):')
    display(numeric_distribution_report.head(30))
    if not numeric_distribution_report.empty:
        display(numeric_distribution_report.groupby('Attack', dropna=False)[
            'KS_D'
        ].agg(['median', 'max']))
    print('Categorical distribution differences (total variation distance):')
    display(categorical_distribution_report)
else:
    print('Within-class distribution comparisons disabled.')

# Release the full original before encoding, splitting, and preprocessing.
# The sampled df and the small comparison reports remain available.
del original_df
gc.collect()
print('Sampling comparison finished; original data released from RAM.')

## 2. Encoding

The sets below control the `categories` branch. All learned encoders and PCA are fitted on training data only. Keep their raw inputs until transformation.

| Feature | Handling |
|---|---|
| `PROTOCOL`, `L7_PROTO`, `ICMP_IPV4_TYPE`, `DNS_QUERY_TYPE`, `FTP_COMMAND_RET_CODE` | One-hot encoding |
| `IPV4_SRC_ADDR`, `IPV4_DST_ADDR` | Integer encoding, then PCA jointly with `DNS_QUERY_ID` |
| `DNS_QUERY_ID` | Combined with encoded IPs, scaled, then PCA |
| `L4_SRC_PORT`, `L4_DST_PORT` | Binary category encoding |
| `TCP_FLAGS`, `CLIENT_TCP_FLAGS`, `SERVER_TCP_FLAGS` | Eight flag indicators per field |
| `ICMP_TYPE` | Decode the code and one-hot encode it; retain the existing `ICMP_IPV4_TYPE` |

The earlier NF-ToN-IoT check found exact agreement between `ICMP_IPV4_TYPE` and the decoded type on every row. The optional check below can be repeated when changing source datasets. Identifier PCA is an experimental representation: integer IP codes have arbitrary geometry, and PCA does not remove network identity information.

`Encoding(..., scaler=IDENTIFIER_SCALER)` accepts an unfitted scaler instance such as `StandardScaler()`, `MinMaxScaler()` or `QuantileTransformer(output_distribution='normal', random_state=42)`. It scales the three identifier inputs **before PCA**, using its own cloned estimator. One-hot, binary-port and flag outputs are unchanged. `NUMERIC_SCALER` independently controls measurement scaling after imputation; set both variables to the same scaler type if desired. Scalers, learned encoders and PCA are fitted using the full training split. Only IterativeImputer is fitted on a capped training sample, then frozen. QuantileTransformer may internally subsample when estimating quantiles.

`IDENTIFIER_COMPONENTS = 1` compresses the two integer-encoded IPs and `DNS_QUERY_ID` jointly to one output, `categories__pca__pca0`. The fit cell prints the selected scalers and retained variance. Change the component count to 2 or 3 to retain more information. Retained variance refers to the **scaled identifier space**, not attack-prediction quality. Integer IP codes have arbitrary order and DNS query IDs are transaction identifiers, so validate this representation against dropping these identifiers.

MinMax scaling changes ranges but does not reduce outlier influence. Quantile transformation can reduce marginal outlier influence but changes distances/correlations and maps values outside the fitted range to distribution bounds. Scaling before PCA does not bound the final PCA scores. The numeric branch applies log1p to SRC_TO_DST_SECOND_BYTES and DST_TO_SRC_SECOND_BYTES before imputation, then applies the selected measurement scaler. The same fixed compression is used for the imputer fit sample and for every raw-data transformation. It retains rows and ordering, but does not establish whether the original extreme values are valid.

References: [PCA](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html), [MinMaxScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html), [QuantileTransformer](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.QuantileTransformer.html).


### Flag decomposition

In [ ]:
df['TCP_FLAGS'].value_counts()

In [ ]:
df.CLIENT_TCP_FLAGS.value_counts()

In [ ]:
import pandas as pd

def decompose_tcp_flags(
    df,
    columns=('TCP_FLAGS', 'CLIENT_TCP_FLAGS', 'SERVER_TCP_FLAGS'),
    inplace=False,
):
    flags = {
        'FIN': 1,
        'SYN': 2,
        'RST': 4,
        'PSH': 8,
        'ACK': 16,
        'URG': 32,
        'ECE': 64,
        'CWR': 128,
    }

    new_columns = {}

    for column in columns:
        values = pd.to_numeric(df[column], errors='raise')

        # Reject missing, fractional, or out-of-range values.
        if (
            values.isna().any()
            or not values.between(0, 255).all()
            or (values % 1 != 0).any()
        ):
            raise ValueError(
                f'{column} must contain non-missing integers from 0 to 255.'
            )

        values = values.to_numpy(dtype='int64')

        for flag, mask in flags.items():
            new_columns[f'{column}_{flag}'] = (
                (values & mask) != 0
            ).astype('uint8')

    result = pd.DataFrame(new_columns, index=df.index)

    if inplace:
        df[result.columns] = result
        return None

    return result

In [ ]:
# Flag decomposition is called inside Encoding; do not modify df here.

### ICMP_TYPE decoding into Type and code
* Check reduncy with ICMP_IPV4_TYPE

In [ ]:
import pandas as pd

def decode_icmp_type(df, column='ICMP_TYPE', inplace=False):
    values = pd.to_numeric(df[column], errors='raise')

    if (
        values.isna().any()
        or not values.between(0, 65535).all()
        or (values % 1 != 0).any()
    ):
        raise ValueError(
            f'{column} must contain non-missing integers from 0 to 65535.'
        )

    values = values.to_numpy(dtype='int64')

    result = pd.DataFrame({
        'ICMP_TYPE_DECODED': (values // 256).astype('uint8'),
        'ICMP_CODE_DECODED': (values % 256).astype('uint8'),
    }, index=df.index)

    if inplace:
        df[result.columns] = result
        return None

    return result

In [ ]:
# Optional diagnostic; the model pipeline decodes ICMP independently.
CHECK_ICMP_REDUNDANCY = False

if CHECK_ICMP_REDUNDANCY:
    icmp_df = decode_icmp_type(df)
    display(icmp_df['ICMP_TYPE_DECODED'].value_counts())

In [ ]:
if CHECK_ICMP_REDUNDANCY:
    original_type = pd.to_numeric(df['ICMP_IPV4_TYPE'], errors='raise')
    decoded_type = icmp_df['ICMP_TYPE_DECODED']
    valid = original_type.notna() & decoded_type.notna()
    matches = decoded_type.loc[valid].eq(original_type.loc[valid])

    print('Compared rows:', valid.sum())
    print('Excluded missing rows:', (~valid).sum())
    print('Mismatches:', (~matches).sum())
    if len(matches):
        print(f'Match percentage: {matches.mean():.2%}')
    print('Encoding retains ICMP_IPV4_TYPE and extracts the code from ICMP_TYPE.')

In [ ]:
if CHECK_ICMP_REDUNDANCY:
    del icmp_df, original_type, decoded_type, valid, matches
# ICMP_TYPE and ICMP_IPV4_TYPE remain available to Encoding.

In [ ]:
# Add only unrelated columns you want removed before splitting.
# Keep identifiers, raw flags, and ICMP inputs for Encoding.
col_to_drop = []

In [ ]:
col_to_one_hot = [
    'PROTOCOL',
    'L7_PROTO',
    'ICMP_IPV4_TYPE',
    'DNS_QUERY_TYPE',
    'FTP_COMMAND_RET_CODE',
]

col_to_binary = ['L4_SRC_PORT', 'L4_DST_PORT']
col_to_label = ['IPV4_SRC_ADDR', 'IPV4_DST_ADDR']
col_to_pca = col_to_label + ['DNS_QUERY_ID']

col_to_decompose = [
    'TCP_FLAGS',
    'CLIENT_TCP_FLAGS',
    'SERVER_TCP_FLAGS',
]

FLAG_NAMES = ['FIN', 'SYN', 'RST', 'PSH', 'ACK', 'URG', 'ECE', 'CWR']
IDENTIFIER_COMPONENTS = 1  # Three identifier inputs -> one PCA output column.

# Pass scaler INSTANCES. Each pipeline gets its own unfitted copy.
NUMERIC_SCALER = QuantileTransformer(output_distribution='normal', random_state=SEED)  # Measurements, after imputation.
IDENTIFIER_SCALER = QuantileTransformer(output_distribution='normal', random_state=SEED)  # Encoded IPs + DNS_QUERY_ID, before PCA.
# Alternatives for either setting:
# MinMaxScaler()
# QuantileTransformer(output_distribution='normal', random_state=42)
# QuantileTransformer(output_distribution='uniform', random_state=42)
IMPUTER_FIT_ROWS = 500_000  # Sample cap applies only to IterativeImputer.

# Progress and reporting only; no category coverage is enforced.
PREPROCESSING_VERBOSE = True
RUN_CATEGORY_COVERAGE_TESTS = True
PRINT_EVERY_BATCHES = 10  # Also print the first and last batch of each split.

In [ ]:
# These are retained as inputs to the identifier PCA branch.
identifiers = col_to_pca.copy()

In [ ]:
col_categorical = (
    col_to_one_hot
    + col_to_binary
    + col_to_pca
    + col_to_decompose
    + ['ICMP_TYPE']
)

In [ ]:
def prepare_categories(X, flag_cols):
    X = X.copy()
    decompose_tcp_flags(X, columns=flag_cols, inplace=True)
    X['ICMP_CODE_DECODED'] = decode_icmp_type(X)['ICMP_CODE_DECODED']
    return X

def prepared_category_names(transformer, input_features):
    flag_cols = transformer.kw_args['flag_cols']
    return (
        list(input_features)
        + [f'{col}_{flag}' for col in flag_cols for flag in FLAG_NAMES]
        + ['ICMP_CODE_DECODED']
    )

def Encoding(one_hot, binary, label, pca, flags, n_components=1, verbose=True,
             scaler=None):
    """Use the supplied scaler on identifier inputs before PCA only."""
    identifier_scaler = StandardScaler() if scaler is None else clone(scaler)
    flag_columns = [
        f'{col}_{flag}' for col in flags for flag in FLAG_NAMES
    ]
    identifiers = ColumnTransformer([
        ('labels', OrdinalEncoder(
            handle_unknown='use_encoded_value', unknown_value=-1,
        ), label),
        ('numeric', SimpleImputer(strategy='median'),
         [c for c in pca if c not in label]),
    ], verbose=verbose)
    return make_pipeline(
        FunctionTransformer(
            prepare_categories,
            kw_args={'flag_cols': flags},
            feature_names_out=prepared_category_names,
        ),
        ColumnTransformer([
            ('onehot', OneHotEncoder(handle_unknown='ignore'),
             one_hot + ['ICMP_CODE_DECODED']),
            ('binary', BinaryEncoder(
                cols=binary, handle_unknown='value', handle_missing='value',
            ), binary),
            ('flags', 'passthrough', flag_columns),
            ('pca', make_pipeline(
                identifiers, identifier_scaler,
                PCA(n_components=n_components, svd_solver='full'),
                verbose=verbose,
            ), pca),
        ], verbose=verbose),
        verbose=verbose,
    )

## 3. Dataset Shrinking

In [ ]:
import warnings
from fastai.tabular.core import df_shrink_dtypes

results = []

for col in df.select_dtypes(include="number").columns:
    # These raw values must stay float64 until the numerical pipeline logs them.
    if col in EXTREME_COLUMNS:
        results.append({
            "column": col,
            "original_dtype": str(df[col].dtype),
            "proposed_dtype": "float64 (preserved)",
            "minimum": df[col].min(),
            "maximum": df[col].max(),
            "warning": "Skipped shrinking; log1p is applied before imputation.",
        })
        continue

    # Inspect one column without copying or modifying the DataFrame.
    skip = [c for c in df.columns if c != col]

    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", RuntimeWarning)

        proposed = df_shrink_dtypes(
            df,
            skip=skip,
            obj2cat=False,
            int2uint=False,
        )

    if caught:
        results.append({
            "column": col,
            "original_dtype": str(df[col].dtype),
            "proposed_dtype": str(proposed.get(col, df[col].dtype)),
            "minimum": df[col].min(),
            "maximum": df[col].max(),
            "warning": "; ".join(str(w.message) for w in caught),
        })

display(pd.DataFrame(results))

In [ ]:
big_cols = EXTREME_COLUMNS

for col in big_cols:
    s = df[col]
    print(f"\n{col}")
    print("Outside float32 range:",
          (s.abs() > float(np.finfo(np.float32).max)).sum())
    print(s.quantile([0.5, 0.9, 0.99, 0.999, 1.0]))
    display(df.nlargest(20, col)[
        [col, "IN_BYTES", "OUT_BYTES",
         "FLOW_DURATION_MILLISECONDS", "Attack"]
    ])

In [ ]:
df.columns = df.columns.str.strip()
df['Attack'] = df['Attack'].astype("string[pyarrow]").str.strip()

In [ ]:
# Preserve the raw extreme measurements; compress them inside the preprocessor.
df = df_shrink(df, skip=EXTREME_COLUMNS, obj2cat=False, int2uint=False)
df.dtypes

## 4. Clean

In [ ]:
# Keep all raw encoding inputs, including identifiers.
df = df.drop(columns=['Dataset'], errors='ignore')

In [ ]:
print(df.duplicated().sum(), "fully duplicate rows to remove")

In [ ]:
df.drop_duplicates(inplace=True)
df.reset_index(inplace=True, drop=True)

In [ ]:
# Keep Label and Attack for splitting/evaluation; never use them as features.
df = df.loc[:, ~df.columns.str.lower().str.startswith(('unnamed:', 'index'))]
feature_cols = [c for c in df if c not in ('Label', 'Attack')]
categorical_cols = list(col_categorical)
numeric_cols = [c for c in feature_cols if c not in categorical_cols]

# IPs are strings. Numeric protocol/category codes keep their original types.
for col in col_to_label:
    df[col] = df[col].astype('string[pyarrow]').fillna('missing').str.strip()

df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)
print(f'{len(numeric_cols)} numeric and {len(categorical_cols)} raw encoding inputs')

In [ ]:
print('Categorical: \n', categorical_cols)
print('\n Numeric: \n', numeric_cols)

In [ ]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)
print(df.isna().any(axis=1).sum(), "rows with at least one NaN to remove")
# df.dropna(inplace=True)

## 5. Optionally reserve an unseen attack

In [ ]:
# Remove incomplete targets and exact repeated rows before any split.
if HOLD_OUT_ATTACK:
    zero_day = df[df['Attack'].eq(HELD_OUT_ATTACK)]
    known = df[df['Attack'].ne(HELD_OUT_ATTACK)]
    print(f'Eligible: {len(known):,} | held-out {HELD_OUT_ATTACK}: {len(zero_day):,}')
else:
    known = df
    zero_day = df.iloc[:0]  # Keep the same schema; no zero-day file is saved.
    print(f'Holdout disabled: all {len(known):,} rows enter the normal split.')


## 6. Split eligible traffic (70% / 15% / 15%)

In [ ]:
# Stratify eligible traffic by binary Label; exclude the held-out attack only when enabled.
train, remainder = train_test_split(
    known, test_size=0.30, stratify=known['Attack'], random_state=SEED
)
val, test = train_test_split(
    remainder, test_size=0.50, stratify=remainder['Attack'], random_state=SEED
)
# Drop intermediate frames; retain only the four clean, untransformed splits.
del df, known, remainder
print('train / val / test / zero-day:', *(len(x) for x in (train, val, test, zero_day)))

## Outliers

In [ ]:
# Only numerical measurements—not categories or flag indicators.
outlier_cols = [
    c for c in numeric_cols
    if c not in col_to_one_hot
    and not c.startswith((
        'TCP_FLAGS_',
        'CLIENT_TCP_FLAGS_',
        'SERVER_TCP_FLAGS_',
    ))
]

rows = []

for col in outlier_cols:
    s = train[col]
    finite = s[np.isfinite(s)]

    q1, median, q3, p99, p999 = finite.quantile(
        [0.25, 0.50, 0.75, 0.99, 0.999]
    )
    iqr = q3 - q1

    # Zero IQR makes this rule uninformative for sparse/zero-heavy features.
    iqr_count = (
        ((finite < q1 - 1.5 * iqr) |
         (finite > q3 + 1.5 * iqr)).sum()
        if iqr > 0 else np.nan
    )

    rows.append({
        'feature': col,
        'missing': s.isna().sum(),
        'infinite': np.isinf(s).sum(),
        'minimum': finite.min(),
        'median': median,
        'p99': p99,
        'p99.9': p999,
        'maximum': finite.max(),
        'IQR': iqr,
        'IQR_flagged': iqr_count,
        'above_p99.9': (finite > p999).sum(),
        'outside_float32': (
            (finite > np.finfo(np.float32).max) |
            (finite < -np.finfo(np.float32).max)
        ).sum(),
        'outside_float64': (
            (finite > np.finfo(np.float64).max) |
            (finite < -np.finfo(np.float64).max)
        ).sum(),
    })

outlier_report = pd.DataFrame(rows).set_index('feature')
display(outlier_report)

## 8. Fit preprocessing on training data only
Apply fixed log1p compression to the two SECOND_BYTES measurements before imputation. No rows are removed or percentile limits fitted. Fit IterativeImputer with `skip_complete=True` on at most `IMPUTER_FIT_ROWS` training rows, then wrap it in `FrozenEstimator` so the main fit cannot refit it. Fit the remaining scalers, encoders and identifier PCA using the full training split. Requires scikit-learn 1.6 or newer.

Run from Setup after this change. Old saved preprocessors and prepared datasets must be regenerated.

The main fit still transforms all training rows in memory, including imputation and encoded features. The batch-saving stage below does not batch this fitting step. Category coverage checks remain reports only and never change the split.


In [ ]:
from time import perf_counter
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.frozen import FrozenEstimator

print(f'Preparing imputer sample from {len(train):,} training rows...', flush=True)
imputer_sample = train.sample(
    n=min(IMPUTER_FIT_ROWS, len(train)), random_state=SEED,
)[numeric_cols]
imputer_fit_rows = len(imputer_sample)
imputer_sample = compress_extreme_values(imputer_sample)

imputer = IterativeImputer(
    max_iter=5, random_state=SEED, skip_complete=True,
    verbose=int(PREPROCESSING_VERBOSE)
)
print(f'Fitting ONLY IterativeImputer on {imputer_fit_rows:,} rows...', flush=True)
imputer_started = perf_counter()
imputer.fit(imputer_sample)
print(f'Imputer fitted in {perf_counter() - imputer_started:.1f}s; '
      'freezing it before the remaining fit.', flush=True)
del imputer_sample

preprocessor = ColumnTransformer([
    ('numbers', make_pipeline(
        FunctionTransformer(
            compress_extreme_values, feature_names_out='one-to-one',
        ),
        FrozenEstimator(imputer), clone(NUMERIC_SCALER), verbose=PREPROCESSING_VERBOSE
    ), numeric_cols),
    ('categories', Encoding(
        one_hot=col_to_one_hot,
        binary=col_to_binary,
        label=col_to_label,
        pca=col_to_pca,
        flags=col_to_decompose,
        n_components=IDENTIFIER_COMPONENTS,
        scaler=IDENTIFIER_SCALER,
        verbose=PREPROCESSING_VERBOSE,
    ), categorical_cols),
], verbose=PREPROCESSING_VERBOSE)

print('Measurement scaler:', NUMERIC_SCALER, flush=True)
print('Identifier scaler before PCA:', IDENTIFIER_SCALER, flush=True)
print(f'Identifier PCA: {len(col_to_pca)} inputs -> {IDENTIFIER_COMPONENTS} component(s).', flush=True)

preprocessor_fit_rows = len(train)
print(f'Fitting scalers, encoders and PCA using all {preprocessor_fit_rows:,} training rows: '
      f'{len(numeric_cols)} numeric columns and '
      f'{len(categorical_cols)} encoding inputs. Imputer stays frozen.', flush=True)
fit_started = perf_counter()
preprocessor.fit(train)
print(f'Preprocessing fit finished in {perf_counter() - fit_started:.1f}s. '
      f'Output features: {len(preprocessor.get_feature_names_out()):,}.', flush=True)
# Diagnostic only: variance is measured after the selected identifier scaling.
fitted_identifier_pipeline = (
    preprocessor.named_transformers_['categories'].steps[-1][1]
    .named_transformers_['pca']
)
fitted_identifier_pca = fitted_identifier_pipeline.steps[-1][1]
identifier_variance_retained = fitted_identifier_pca.explained_variance_ratio_.sum()
print('Identifier PCA variance per component:',
      fitted_identifier_pca.explained_variance_ratio_, flush=True)
print(f'Identifier PCA retains {identifier_variance_retained:.2%} of the scaled identifier variance.',
      flush=True)


### TEST — category coverage (report only)

Compare all observed unique values across train/validation/test (and optional zero-day) with the full training split and the fitted one-hot/binary/IP vocabularies. This also checks the generated ICMP code. Prints and results do not enforce coverage or change the split. `False` means some values are absent; example values are included. Raw flags, packed ICMP and numeric DNS ID are reported for split/sample coverage but have no learned category vocabulary.

Counts describe the current cleaned/split dataset. Checking evaluation vocabularies is diagnostic only; do not use these results to refit vocabularies on evaluation data. Set `RUN_CATEGORY_COVERAGE_TESTS = False` to skip the full-column scan.

Encoders now fit using every training row. The separate numeric imputer sample does not limit category vocabularies.


In [ ]:
# Reporting only: never resplit, resample, refit, or move rows.
if RUN_CATEGORY_COVERAGE_TESTS:
    def category_values(values):
        values = pd.Series(values, copy=False)
        result = set(values.dropna().unique().tolist())
        if values.isna().any():
            result.add(('<missing>',))
        return result

    # Inspect the vocabularies actually learned by the fitted encoders.
    fitted_encoding = preprocessor.named_transformers_['categories'].steps[-1][1]
    learned_values = dict(zip(
        col_to_one_hot + ['ICMP_CODE_DECODED'],
        fitted_encoding.named_transformers_['onehot'].categories_,
    ))
    for mapping in fitted_encoding.named_transformers_['binary'].ordinal_encoder.category_mapping:
        positive_codes = mapping['mapping'] > 0
        learned_values[mapping['col']] = mapping['mapping'].index[positive_codes]
    fitted_ips = fitted_encoding.named_transformers_['pca'].steps[0][1].named_transformers_['labels']
    learned_values.update(dict(zip(col_to_label, fitted_ips.categories_)))

    coverage_splits = {'train': train, 'val': val, 'test': test}
    if HOLD_OUT_ATTACK:
        coverage_splits['zero_day'] = zero_day

    coverage_rows = []
    coverage_columns = categorical_cols + ['ICMP_CODE_DECODED']
    print('Checking full-training and fitted-encoder category coverage one column at a time...', flush=True)

    for number, col in enumerate(coverage_columns, start=1):
        # Decode only the code column; do not build a full transformed dataset.
        if col == 'ICMP_CODE_DECODED':
            split_values = {
                name: category_values(rows['ICMP_TYPE'] % 256)
                for name, rows in coverage_splits.items()
            }
        else:
            split_values = {
                name: category_values(rows[col])
                for name, rows in coverage_splits.items()
            }

        all_values = set().union(*split_values.values())
        train_values = split_values['train']
        missing_train = all_values - train_values
        learned = category_values(learned_values[col]) if col in learned_values else None
        missing_encoder = all_values - learned if learned is not None else None

        coverage_rows.append({
            'column': col,
            'all_unique': len(all_values),
            'train_unique': len(train_values),
            'missing_from_train': len(missing_train),
            'train_covers_all': not missing_train,
            'learned_unique': len(learned) if learned is not None else pd.NA,
            'missing_from_encoder': len(missing_encoder) if learned is not None else pd.NA,
            'encoder_covers_all': not missing_encoder if learned is not None else pd.NA,
            'val_unseen_by_encoder': len(split_values['val'] - learned) if learned is not None else pd.NA,
            'test_unseen_by_encoder': len(split_values['test'] - learned) if learned is not None else pd.NA,
            'zero_day_unseen_by_encoder': len(split_values.get('zero_day', set()) - learned) if learned is not None else pd.NA,
            'missing_train_examples': repr(sorted(missing_train, key=str)[:5]),
            'missing_encoder_examples': repr(sorted(missing_encoder, key=str)[:5]) if learned is not None else '',
        })
        encoder_status = ('not a category vocabulary' if learned is None else
                          f'{len(missing_encoder):,} values absent from fitted encoder')
        print(f'[{number}/{len(coverage_columns)}] {col}: '
              f'train {len(train_values):,}/{len(all_values):,} unique values; '
              f'{encoder_status}.', flush=True)

    category_coverage_report = pd.DataFrame(coverage_rows).set_index('column')
    display(category_coverage_report)
    training_complete = bool(category_coverage_report['train_covers_all'].all())
    encoder_results = category_coverage_report['encoder_covers_all'].dropna()
    encoders_complete = bool(encoder_results.all())
    print('Full training split covers every reported unique value:', training_complete)
    print('Fitted categorical vocabularies cover every observed category:', encoders_complete)
    print('Missing values, if present, are counted as one category.')
    print('Flag combinations and packed ICMP are decoded by fixed rules; '
          'DNS_QUERY_ID enters PCA numerically, so these rows have no learned category vocabulary.')
    if HOLD_OUT_ATTACK:
        print('All_unique includes the held-out attack set; unseen categories there may be intentional.')
    print('Report only: coverage failures do not stop execution or change any split.')
    del coverage_splits



Save each split below in batches, then release its original frame. To repeat saving or run the optional raw-data benchmarks afterward, rerun the loading/splitting cells first.

## 9. Transform, save, and free each split

In [ ]:
import pyarrow as pa

feature_names = preprocessor.get_feature_names_out().tolist()

binary_prefixes = (
    'categories__onehot__',
    'categories__binary__',
    'categories__flags__',
)

float64_columns = {
    'numbers__SRC_TO_DST_SECOND_BYTES',
    'numbers__DST_TO_SRC_SECOND_BYTES',
}

original_target_schema = pa.Schema.from_pandas(
    train[['Label', 'Attack']], preserve_index=False
)

schema = pa.schema(
    [
        (
            name,
            pa.uint8() if name.startswith(binary_prefixes)
            else pa.float64() if name in float64_columns
            else pa.float32()
        )
        for name in feature_names
    ]
    + [
        original_target_schema.field('Label'),
        original_target_schema.field('Attack'),
    ]
)

print(f'Compact output schema:\n{schema}')

import shutil
shutil.rmtree('/kaggle/working/project')

In [ ]:
from datetime import datetime
from time import perf_counter
import gc
import joblib
import sklearn
import pyarrow as pa
import pyarrow.parquet as pq
from scipy.sparse import issparse

PROJECT = Path(OUTPATH)
fitted_imputer = preprocessor.named_transformers_['numbers'].named_steps['frozenestimator'].estimator
fitted_numeric_scaler = preprocessor.named_transformers_['numbers'].steps[-1][1]
fitted_identifier_pipeline = (
    preprocessor.named_transformers_['categories'].steps[-1][1]
    .named_transformers_['pca']
)
fitted_identifier_scaler = fitted_identifier_pipeline.steps[-2][1]
fitted_identifier_pca = fitted_identifier_pipeline.steps[-1][1]
attack_tag = f"holdout_{HELD_OUT_ATTACK}" if HOLD_OUT_ATTACK else "all_attacks"
RUN_NAME = f"{datetime.now():%Y%m%d_%H%M%S}_{attack_tag}_log1p_{type(fitted_imputer).__name__}"
OUTPUT_DIR = PROJECT / 'outputs' / DATASET / 'preprocessing' / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BATCH_SIZE = 10000  # Reduce this if a transformed batch still uses too much RAM.
print('Saving fitted preprocessor to:', OUTPUT_DIR, flush=True)
joblib.dump(preprocessor, OUTPUT_DIR / 'preprocessor.joblib', compress=0)
print('Fitted preprocessor saved.', flush=True)

# Move frames into the queue; remove the other references so they can be freed.
splits = {'train': train, 'val': val, 'test': test}
if HOLD_OUT_ATTACK:
    splits['zero_day'] = zero_day
del train, val, test, zero_day
counts = []
for name in list(splits):
    rows = splits.pop(name)
    split_started = perf_counter()
    total_batches = (len(rows) + BATCH_SIZE - 1) // BATCH_SIZE
    print(f'Starting {name}: {len(rows):,} rows in {total_batches:,} batches.', flush=True)
    with pq.ParquetWriter(OUTPUT_DIR / f'{name}.parquet', schema, compression='zstd') as writer:
        for start in range(0, len(rows), BATCH_SIZE):
            batch = rows.iloc[start:start + BATCH_SIZE]
            X = preprocessor.transform(batch[feature_cols])
            # # Reduce batch memory before densifying sparse output.
            # # float32 rounding is intentional for compact model inputs.
            # X = X.astype(np.float32, copy=False)
            # Densify this small batch only; never the complete split.
            if issparse(X):
                X = X.toarray()
            prepared = pd.DataFrame(X, columns=feature_names)
            prepared['Label'] = batch['Label'].to_numpy()
            prepared['Attack'] = batch['Attack'].to_numpy()
            table = pa.Table.from_pandas(prepared, schema=schema, preserve_index=False)
            writer.write_table(table)
            batch_number = start // BATCH_SIZE + 1
            if (batch_number == 1 or batch_number == total_batches
                    or batch_number % max(1, PRINT_EVERY_BATCHES) == 0):
                completed = min(start + BATCH_SIZE, len(rows))
                print(f'{name}: batch {batch_number:,}/{total_batches:,}; '
                      f'{completed:,}/{len(rows):,} rows written; '
                      f'{perf_counter() - split_started:.1f}s elapsed.', flush=True)
            del batch, X, prepared, table
    counts.append(f'- {name}: {len(rows):,} rows')
    print(f'Saved {name}: {len(rows):,} rows in {perf_counter() - split_started:.1f}s; releasing raw split.', flush=True)
    del rows
    gc.collect()  # Frees Python references; the allocator may retain some reserved RAM.

holdout_note = f"Reserve `{HELD_OUT_ATTACK}` as zero-day data" if HOLD_OUT_ATTACK else "Use all attack classes; no separate zero-day set"
saved_files = '`train.parquet`, `val.parquet`, `test.parquet`' + (', `zero_day.parquet`' if HOLD_OUT_ATTACK else '')
readme = f"""# {DATASET} preprocessing — {RUN_NAME}

Source: `{DATA_DIR / FILES[DATASET]}`
Holdout enabled: `{HOLD_OUT_ATTACK}`. Held-out attack: `{HELD_OUT_ATTACK if HOLD_OUT_ATTACK else "None (all attacks included)"}`. Seed: `{SEED}`. scikit-learn: `{sklearn.__version__}`.

## Steps followed
1. Inspect data and attack counts; strip column/attack-name whitespace.
2. Remove Dataset, configured drop columns, and generated index columns; retain IPs and DNS_QUERY_ID for identifier PCA.
3. Keep categorical numeric codes separate from measurements; normalize IP strings and replace numeric infinities with NaN.
4. Remove exact duplicate rows.
5. {holdout_note}; split eligible traffic 70/15/15, stratified by Attack. With holdout disabled, attack classes are eligible for every split; rare classes are not guaranteed to occur in each split.
6. Apply fixed log1p compression to `{EXTREME_COLUMNS}` before imputation, without clipping or deleting rows. Fit only `{fitted_imputer!r}` on {imputer_fit_rows:,} sampled training rows (cap {IMPUTER_FIT_ROWS:,}), then freeze it. Fit the remaining preprocessing using all {preprocessor_fit_rows:,} training rows: `{fitted_numeric_scaler!r}` for measurements; Encoding combines one-hot codes, binary ports, decomposed TCP flags, decoded ICMP code, and integer IP encoding plus identifier scaling with `{fitted_identifier_scaler!r}` then PCA ({fitted_identifier_pca.n_components_} components).
7. Transform each split in batches of at most {BATCH_SIZE:,} rows and write directly to Parquet with Zstandard compression.
8. Release each batch and completed split before continuing.

## Identifier PCA diagnostics
Input columns: `{col_to_pca}`. Retained variance after identifier scaling: `{fitted_identifier_pca.explained_variance_ratio_.sum():.2%}`.
Variance is a compression diagnostic, not a measure of attack-prediction quality. IP ordinal mappings are arbitrary and DNS query IDs are transaction identifiers.
Scaling before PCA does not bound PCA outputs. The two SECOND_BYTES measurements are log1p-compressed before imputation. Measurement scaling follows imputation.

## Progress and category coverage
PREPROCESSING_VERBOSE controls fit-step timings. PRINT_EVERY_BATCHES controls batch-write progress.
RUN_CATEGORY_COVERAGE_TESTS enables a report comparing full-training/evaluation vocabularies and actual fitted encoders. Only the numeric imputer uses a training sample; encoders use the full training split.
Checks do not enforce coverage, refit on evaluation data, resample, or modify any split.

## Saved files
{saved_files} contain TRANSFORMED numeric feature columns plus `Label` and `Attack`.
Features are already log-compressed where configured, imputed, encoded, and scaled: do not transform them again.
`preprocessor.joblib` is for new RAW data only. Feature order is `preprocessor.get_feature_names_out()`.
Storage types: one-hot, binary-port and flag outputs use uint8; identifier PCA and ordinary scaled measurements use float32. The two configured SECOND_BYTES outputs retain float64 storage.
Label and Attack retain their original Arrow types. The same schema is used for every batch and split.
Float32 reduces precision compared with float64.
These files differ from older exports of cleaned, untransformed features.
To load preprocessor.joblib in a new session, first run the imports, EXTREME_COLUMNS and compress_extreme_values definition, decoder definitions, column sets and Encoding helper definitions; do not fit again.
IP code mappings and PCA are learned from training only. Identifier PCA does not guarantee generalization across networks.
The two SECOND_BYTES features use fixed log1p compression before imputation. This prevents their original magnitudes from reaching the regression models, but does not prove that the original measurements are valid. No outlier rows are removed. Complete columns are skipped by IterativeImputer during fitting.

{chr(10).join(counts)}

## Read prepared batches
```python
import pyarrow.parquet as pq
for record_batch in pq.ParquetFile('train.parquet').iter_batches(batch_size=10000):
    batch = record_batch.to_pandas()
    X = batch.drop(columns=['Label', 'Attack'])
    y = batch['Label']
    # Consume this batch directly; do not apply preprocessor.transform(X).
```

Initial loading/splitting and imputer fitting still require RAM. Fitting the remaining preprocessing transforms the full training split in memory; the later batch-saving loop does not batch fitting. FrozenEstimator requires scikit-learn 1.6 or newer. Saving releases the original split frames; rerun loading/splitting before using raw-data benchmarks or repeating the save step.
Row splits and exact deduplication do not guarantee session isolation or isolation of identical predictors with conflicting labels.
"""
(OUTPUT_DIR / 'README.md').write_text(readme, encoding='utf-8')
print('Prepared datasets saved to:', OUTPUT_DIR)

## 7. Class Imbalance

Main order: split → log1p on the two SECOND_BYTES fields → impute/encode/scale → transform/save → ADASYN on train → Tomek Links → model training.

Load the processed `train.parquet`, resample its `Attack` classes with ADASYN, then clean the original majority Attack class with Tomek Links. Exclude both targets from features. Restore binary `Label` from the original Attack-to-Label mapping when saving. Validation and test are not resampled here. Synthetic one-hot coordinates can be fractional and remain floating point.

In [ ]:
HANDEL_IMBALANCE = True
ADASYN_BALANCE_TOLERANCE = 0.01  # Leave classes within 1% unchanged.

from contextlib import contextmanager
from time import perf_counter

@contextmanager
def resampling_status(stage):
    """Report stage start and total elapsed time without periodic messages."""
    started = perf_counter()
    print(f'{stage}: starting...', flush=True)
    finished = False
    try:
        yield
        finished = True
    finally:
        status = 'finished' if finished else 'stopped before completion'
        print(f'{stage}: {status} | {perf_counter() - started:.1f}s elapsed',
              flush=True)


In [ ]:
import pandas as pd
import pyarrow.parquet as pq

stored_schema = pq.read_schema(OUTPUT_DIR / 'train.parquet')

dtype_map = {
    field.name: field.type.to_pandas_dtype()
    for field in stored_schema
}

display(
    pd.Series(dtype_map, name='dtype')
    .astype(str)
    .rename_axis('column')
    .to_frame()
)

In [ ]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

train_path = OUTPUT_DIR / 'train.parquet'
saved_columns = set(pq.read_schema(train_path).names)

rows = []

for original_col in outlier_cols:
    original_col = original_col.removeprefix('numbers__')

    # Exclude binary flag indicators.
    if original_col.startswith((
        'TCP_FLAGS_',
        'CLIENT_TCP_FLAGS_',
        'SERVER_TCP_FLAGS_',
    )):
        continue

    col = f'numbers__{original_col}'

    if col not in saved_columns:
        print(f'Skipped: {col} is not in the prepared training file.')
        continue

    # Load only this feature, rather than all 486 columns.
    s = pd.read_parquet(train_path, columns=[col])[col]

    # Convert only this column's finite values for the calculations.
    finite = s[np.isfinite(s)].astype(np.float64, copy=False)

    q1, median, q3, p99, p999 = finite.quantile(
        [0.25, 0.50, 0.75, 0.99, 0.999]
    )
    iqr = q3 - q1

    iqr_count = (
        ((finite < q1 - 1.5 * iqr) |
         (finite > q3 + 1.5 * iqr)).sum()
        if iqr > 0 else np.nan
    )

    rows.append({
        'feature': col,
        'dtype': str(s.dtype),
        'missing': s.isna().sum(),
        'infinite': np.isinf(s).sum(),
        'minimum': finite.min(),
        'median': median,
        'p99': p99,
        'p99.9': p999,
        'maximum': finite.max(),
        'IQR': iqr,
        'IQR_flagged': iqr_count,
        'above_p99.9': (finite > p999).sum(),
        'outside_float32': (
            (finite > np.finfo(np.float32).max) |
            (finite < -np.finfo(np.float32).max)
        ).sum(),
        'outside_float64': (
            (finite > np.finfo(np.float64).max) |
            (finite < -np.finfo(np.float64).max)
        ).sum(),
    })

    del s, finite

outlier_report = pd.DataFrame(rows).set_index('feature')
display(outlier_report)

In [ ]:
if HANDEL_IMBALANCE:
    %pip install -q imbalanced-learn plotly

    import gc
    import numpy as np
    import pandas as pd
    import pyarrow.parquet as pq
    import plotly.express as px
    import psutil
    from imblearn.over_sampling import ADASYN
    from imblearn.under_sampling import TomekLinks

    parquet = pq.ParquetFile(OUTPUT_DIR / 'train.parquet')
    stored_schema = parquet.schema_arrow

    original_columns = stored_schema.names
    resampling_features = [
        c for c in original_columns if c not in ('Label', 'Attack')
    ]

    n_rows = parquet.metadata.num_rows
    n_features = len(resampling_features)

    # Read each feature's original dtype from the saved schema.
    dtype_map = {
        column: np.dtype(
            stored_schema.field(column).type.to_pandas_dtype()
        )
        for column in resampling_features
    }

    # Feature storage if each column keeps its original dtype.
    schema_feature_bytes = n_rows * sum(
        dtype.itemsize for dtype in dtype_map.values()
    )

    # NumPy requires one common dtype.
    # Include float32 so synthetic fractional values can be represented.
    matrix_dtype = np.result_type(
        np.float32, *dtype_map.values()
    )

    # Actual NumPy matrix plus pointers for the target array.
    required_bytes = n_rows * (
        n_features * matrix_dtype.itemsize
        + np.dtype(object).itemsize
    )
    available_bytes = psutil.virtual_memory().available

    print(f'Rows: {n_rows:,} | Features: {n_features}', flush=True)
    print(
        f'Features with original column types: '
        f'{schema_feature_bytes / 2**30:.2f} GiB',
        flush=True,
    )
    print(f'Working matrix dtype: {matrix_dtype}', flush=True)
    print(
        f'Working arrays: approximately '
        f'{required_bytes / 2**30:.2f} GiB',
        flush=True,
    )
    print(
        f'Available RAM: {available_bytes / 2**30:.2f} GiB',
        flush=True,
    )

    # Estimates exclude attack-string contents, temporary arrays,
    # and the additional memory needed by ADASYN/Tomek.
    if required_bytes > available_bytes * 0.5:
        raise MemoryError(
            'The working arrays would use over half the available RAM. '
            'Reduce the training rows before resampling.'
        )

    X_train = np.empty(
        (n_rows, n_features),
        dtype=matrix_dtype,
    )
    y_train = np.empty(n_rows, dtype=object)

    target_parts = []
    start = 0

    for record_batch in parquet.iter_batches(
        batch_size=20_000,
        use_threads=False,
    ):
        batch = record_batch.to_pandas()
        stop = start + len(batch)

        if start == 0:
            target_dtypes = batch[['Label', 'Attack']].dtypes

        # Fill the matrix without another complete numeric copy.
        for j, column in enumerate(resampling_features):
            X_train[start:stop, j] = batch[column].to_numpy(
                copy=False
            )

        y_train[start:stop] = batch['Attack'].to_numpy(copy=False)

        target_parts.append(
            batch[['Attack', 'Label']].drop_duplicates()
        )

        start = stop
        del batch, record_batch

    target_pairs = pd.concat(
        target_parts, ignore_index=True
    ).drop_duplicates()

    if target_pairs['Attack'].duplicated().any():
        raise ValueError(
            'An Attack class has multiple Label values.'
        )

    attack_to_label = target_pairs.set_index('Attack')['Label']

    before_counts = pd.Series(y_train).value_counts().sort_index()
    majority_attack = before_counts.idxmax()

    del target_parts, target_pairs
    gc.collect()

    print('Loading completed.', flush=True)
    display(
        before_counts.rename_axis('Attack').rename('rows').to_frame()
    )

ADASYN uses an explicit dictionary of desired class counts. Classes already within `ADASYN_BALANCE_TOLERANCE = 0.01` (1%) of the largest class are left unchanged. This avoids requesting a negligible top-up such as 29 samples across 209,971 Reconnaissance rows, where every per-row allocation can round to zero.

The 1% tolerance is an explicit policy, not a requirement of ADASYN. Smaller classes target the largest count; the generated counts remain approximate. There is no silent fallback to another sampler. Independent ADASYN failures, such as unsuitable neighborhoods, remain visible.

### Runtime reporting

ADASYN and Tomek Links do not expose built-in row-progress callbacks, percentages or ETA. Report stage start, finish/interruption and total elapsed seconds only; no periodic messages are printed. Feature/target shapes, post-ADASYN matrix size, added/removed row counts and the timing summary remain available. ADASYN's skipped case is marked explicitly with zero sampler time.

The samplers still run on the full training matrix. No independent resampling batches, changed neighbour search or approximate cleaning are introduced. Tomek Links, the result tables and the Plotly chart are separate cells; rerun the tables/chart without repeating resampling while their small result variables remain in the session.


### ADASYN — timing and output shape


In [ ]:
if HANDEL_IMBALANCE:
    input_train_rows = len(y_train)
    print(f'Before ADASYN: X={X_train.shape}, y={y_train.shape}', flush=True)
    adasyn_seconds = 0.0
    adasyn_skipped = False
    target_count = int(before_counts.max())
    sampling_targets = {
        attack: target_count
        for attack, count in before_counts.items()
        if count < (1 - ADASYN_BALANCE_TOLERANCE) * target_count
    }

    if sampling_targets:
        smallest_target_class = min(
            int(before_counts[attack]) for attack in sampling_targets
        )
        if smallest_target_class < 2:
            raise ValueError(
                'ADASYN needs at least two examples in each targeted Attack class.'
            )

        print('ADASYN desired class counts:', sampling_targets, flush=True)
        adasyn = ADASYN(
            sampling_strategy=sampling_targets,
            n_neighbors=min(5, smallest_target_class - 1),
            random_state=SEED,
        )
        adasyn_started = perf_counter()
        with resampling_status('ADASYN'):
            X_adasyn, y_adasyn = adasyn.fit_resample(X_train, y_train)
        adasyn_seconds = perf_counter() - adasyn_started
    else:
        adasyn_skipped = True
        print('Classes are already approximately balanced; skipping ADASYN.', flush=True)
        X_adasyn, y_adasyn = X_train, y_train

    adasyn_rows = len(y_adasyn)
    print(f'After ADASYN: X={X_adasyn.shape}, y={y_adasyn.shape}', flush=True)
    print(f'ADASYN time: {adasyn_seconds:.1f}s | '
          f'Rows added: {adasyn_rows - input_train_rows:,}', flush=True)
    print(f'Post-ADASYN feature matrix: {X_adasyn.nbytes / 2**30:.2f} GiB | '
          f'dtype={X_adasyn.dtype}', flush=True)

    adasyn_counts = pd.Series(y_adasyn).value_counts().sort_index()
    print('After ADASYN (Attack):')
    display(pd.DataFrame({
        'Before': before_counts,
        'After ADASYN': adasyn_counts,
    }).fillna(0).astype(int).rename_axis('Attack'))

    del X_train, y_train
    gc.collect()

### Tomek Links — timing and output shape


In [ ]:
if HANDEL_IMBALANCE:
    print(f'Before Tomek Links: X={X_adasyn.shape}, y={y_adasyn.shape}', flush=True)
    # Preserve the original policy: clean only the original majority class.
    tomek = TomekLinks(
        sampling_strategy=[majority_attack],
        n_jobs=-1,
    )
    tomek_started = perf_counter()
    with resampling_status('Tomek Links'):
        X_train_balanced, y_train_balanced = tomek.fit_resample(
            X_adasyn, y_adasyn
        )
    tomek_seconds = perf_counter() - tomek_started
    tomek_removed_rows = len(y_adasyn) - len(y_train_balanced)
    print(f'After Tomek Links: X={X_train_balanced.shape}, '
          f'y={y_train_balanced.shape}', flush=True)
    print(f'Tomek Links time: {tomek_seconds:.1f}s | '
          f'Rows removed: {tomek_removed_rows:,}', flush=True)
    after_counts = pd.Series(y_train_balanced).value_counts()

    # Free the expanded input before plotting or exporting the balanced output.
    del X_adasyn, y_adasyn
    gc.collect()


### Resampling results and timings


In [ ]:
if HANDEL_IMBALANCE:
    distribution = pd.DataFrame({
        'Before': before_counts,
        'After ADASYN': adasyn_counts,
        'After Tomek Links': after_counts,
    }).fillna(0).astype(int).rename_axis('Attack')

    distribution['Before (%)'] = (
        100 * distribution['Before'] / distribution['Before'].sum()
    ).round(2)
    distribution['After (%)'] = (
        100 * distribution['After Tomek Links']
        / distribution['After Tomek Links'].sum()
    ).round(2)
    display(distribution)

    resampling_summary = pd.DataFrame([
        {'Stage': 'ADASYN', 'Status': 'skipped' if adasyn_skipped else 'finished',
         'Input rows': input_train_rows, 'Output rows': adasyn_rows,
         'Features': len(resampling_features), 'Seconds': adasyn_seconds},
        {'Stage': 'Tomek Links', 'Status': 'finished',
         'Input rows': adasyn_rows, 'Output rows': adasyn_rows - tomek_removed_rows,
         'Features': len(resampling_features), 'Seconds': tomek_seconds},
    ]).set_index('Stage')
    display(resampling_summary.round({'Seconds': 2}))


### Plot class distribution

Run this cell again to redraw the chart without rerunning Tomek Links.


In [ ]:
if HANDEL_IMBALANCE:
    # Plot aggregated counts instead of passing millions of rows to Plotly.
    plot_counts = distribution[
        ['Before', 'After ADASYN', 'After Tomek Links']
    ].reset_index().melt(
        id_vars='Attack', var_name='Stage', value_name='Rows'
    )
    plot_counts['Attack'] = plot_counts['Attack'].astype(str)
    fig = px.histogram(
        plot_counts,
        x='Attack',
        y='Rows',
        color='Stage',
        histfunc='sum',
        barmode='group',
        title='Attack class distribution before and after resampling',
        labels={'Rows': 'Number of samples'},
        category_orders={'Attack': distribution.index.astype(str).tolist()},
    )
    fig.update_layout(yaxis_title='Number of samples', xaxis_tickangle=-45)
    fig.show()



In [ ]:
if HANDEL_IMBALANCE:
    balanced_train = pd.DataFrame(
        X_train_balanced,
        columns=resampling_features,
    )
    balanced_train['Attack'] = pd.Series(y_train_balanced).astype(
        target_dtypes['Attack']
    )
    balanced_train['Label'] = balanced_train['Attack'].map(
        attack_to_label
    ).astype(target_dtypes['Label'])
    balanced_train = balanced_train[original_columns]

    balanced_path = OUTPUT_DIR / 'train_adasyn_tomek.parquet'
    balanced_train.to_parquet(
        balanced_path,
        index=False,
        compression='zstd',
    )
    print('Saved:', balanced_path)

    del balanced_train, X_train_balanced, y_train_balanced
    gc.collect()

Train using `train_adasyn_tomek.parquet`, with the existing `val.parquet` and `test.parquet`. The saved training file contains both `Attack` and `Label`; use `Attack` as the multiclass target. Exclude both targets from the model inputs. These features are already preprocessed—do not transform them again.

The table and Plotly histograms above show counts before resampling, after ADASYN, and after Tomek Links. Percentages compare the original and final training distributions.

# Testing

Run Setup and Imports (including `compress_extreme_values`) and the configuration cells at the top of the notebook (platform, paths, dataset and seed), then start here. The loading cell applies log1p once to the two SECOND_BYTES fields before any benchmark imputation or scaling. You can also run this section after the main save cell has released its raw splits. Enable the desired tests below, then run the remaining Testing cells in order.

The first loading cell reloads the **same raw Parquet selected by `DATA_DIR / FILES[DATASET]`**, repeats the relevant feature preparation, and recreates the original 70/15/15 split. Only its training partition enters these experiments. It does not require the main fitted preprocessor or the main in-memory splits.

In [ ]:
# Enable either or both groups, then run the loading cell immediately below.
RUN_TESTING = False      # Existing imputer benchmarks.
RUN_COMPARISONS = False # Encoding -> scaler -> imbalance comparisons, including the paper's method.

In [ ]:
if RUN_TESTING or RUN_COMPARISONS:
    from pathlib import Path
    import gc
    import numpy as np
    import pandas as pd
    from sklearn.model_selection import train_test_split

    if not all(name in globals() for name in ('DATA_DIR', 'FILES', 'DATASET', 'SEED', 'HOLD_OUT_ATTACK', 'HELD_OUT_ATTACK')):
        raise RuntimeError('Run the platform/path/dataset configuration cells at the top first. The main preprocessing, fitting and saving cells are not required.')

    # Release previous experiment frames before reloading the full source.
    for old_name in ('testing_df', 'testing_train', 'exp_sample', 'exp_train', 'exp_val',
                     'exp_num_train', 'exp_num_val', 'exp_cat_train', 'exp_cat_val',
                     'X_case_train', 'X_case_val'):
        globals().pop(old_name, None)
    gc.collect()
    testing_path = Path(DATA_DIR) / FILES[DATASET]
    print('Reloading raw dataset for Testing:', testing_path)
    testing_df = pd.read_parquet(testing_path)
    testing_df.columns = testing_df.columns.str.strip()
    if not testing_df.columns.is_unique:
        raise ValueError('Column names must remain unique after stripping whitespace.')
    if not {'Label', 'Attack'}.issubset(testing_df.columns):
        raise ValueError('Testing requires Label and Attack columns.')
    testing_df['Attack'] = testing_df['Attack'].astype('string[pyarrow]').str.strip()
    if testing_df[['Label', 'Attack']].isna().any().any():
        raise ValueError('Missing targets: clean Label/Attack before splitting; targets are never imputed.')
    if not testing_df['Label'].isin([0, 1]).all():
        raise ValueError('The binary comparisons require Label values 0 and 1.')
    print('Raw dataset:', DATASET, testing_df.shape)
    display(testing_df['Attack'].value_counts())

    # Legacy comparison baseline: identifiers are dropped and raw flags/ICMP stay numeric.
    # This does not reproduce the main Encoding branch or overwrite its saved outputs.
    testing_df = testing_df.drop(columns=[
        'Dataset', 'IPV4_SRC_ADDR', 'IPV4_DST_ADDR', 'SRC_IP', 'DST_IP', 'DNS_QUERY_ID'
    ], errors='ignore')
    testing_df = testing_df.loc[:, ~testing_df.columns.str.lower().str.startswith(('unnamed:', 'index'))]
    testing_feature_cols = [c for c in testing_df if c not in ('Label', 'Attack')]
    testing_categorical_cols = [c for c in ('PROTOCOL', 'L7_PROTO', 'DNS_QUERY_TYPE', 'FTP_COMMAND_RET_CODE')
                                if c in testing_feature_cols]
    for col in testing_categorical_cols:
        testing_df[col] = testing_df[col].astype('string').fillna('missing').astype('string[pyarrow]').str.strip()
    testing_numeric_cols = [c for c in testing_feature_cols if c not in testing_categorical_cols]
    testing_df[testing_numeric_cols] = testing_df[testing_numeric_cols].replace([np.inf, -np.inf], np.nan)
    testing_df = compress_extreme_values(testing_df)
    if HOLD_OUT_ATTACK:
        testing_df = testing_df.loc[testing_df['Attack'].ne(HELD_OUT_ATTACK)]
    if testing_df['Attack'].value_counts().min() < 2:
        raise ValueError('An eligible Attack class has fewer than two rows; stratification is not possible.')
    testing_train, testing_remainder = train_test_split(
        testing_df, test_size=0.30, stratify=testing_df['Attack'], random_state=SEED
    )
    # Do not load/use the original validation, test or held-out partition in these comparisons.
    del testing_df, testing_remainder
    gc.collect()
    print(f'Testing training pool: {len(testing_train):,} rows; original 30% reserved for validation/test.')
    print('Testing numeric/categorical features:', len(testing_numeric_cols), len(testing_categorical_cols))

In [ ]:
if RUN_TESTING:
    import numpy as np
    import pandas as pd
    from sklearn.experimental import enable_iterative_imputer
    from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer
    from sklearn.metrics import mean_absolute_error
    from sklearn.preprocessing import StandardScaler

    print("Starting imputation benchmark on a 50k sample...")

    # 1. Take a 50k sample of the numerical features from the training set
    sample_df = testing_train[testing_numeric_cols].sample(n=min(50000, len(testing_train)), random_state=SEED).copy()

    # Drop any rows that already contain NaNs in the original dataset so we have a clean ground truth
    ground_truth_df = sample_df.dropna()
    print(f"Rows with complete data for ground truth: {len(ground_truth_df)}")

    # Scale the ground truth data first so that features with massive ranges don't dominate the error calculation
    scaler = StandardScaler()
    ground_truth_scaled = pd.DataFrame(scaler.fit_transform(ground_truth_df), columns=testing_numeric_cols)

    # 2. Artificially introduce 10% missing values (NaNs) into the scaled ground truth
    np.random.seed(SEED)
    missing_mask = np.random.rand(*ground_truth_scaled.shape) < 0.10
    masked_data = ground_truth_scaled.copy()
    masked_data[missing_mask] = np.nan

    # We only want to calculate the error on the exact cells we masked
    original_values = ground_truth_scaled.values[missing_mask]

    # 3. Define the Imputers
    imputers = {
        'SimpleImputer (Median)': SimpleImputer(strategy='median'),
        'KNNImputer (k=5, weighted)': KNNImputer(n_neighbors=5, weights='distance'),
        'IterativeImputer (MICE)': IterativeImputer(max_iter=10, random_state=SEED)
    }

    # 4. Evaluate each Imputer
    results = {}
    for name, imputer in imputers.items():
        print(f"Evaluating {name}...")
        try:
            imputed_data = imputer.fit_transform(masked_data)
            imputed_values = imputed_data[missing_mask]
            mae = mean_absolute_error(original_values, imputed_values)
            results[name] = mae
            print(f"{name} MAE: {mae:.5f}")
        except Exception as e:
            results[name] = f"Failed: {str(e)}"
            print(f"{name} failed with error: {e}")

    # 5. Summary of Results
    print("\n--- Benchmark Results (Lower MAE is better) ---")
    for name, score in results.items():
        if isinstance(score, float):
            print(f"* {name}: {score:.5f}")
        else:
            print(f"* {name}: {score}")

In [ ]:
if RUN_TESTING:
    from sklearn.impute import KNNImputer
    from sklearn.preprocessing import StandardScaler, OneHotEncoder
    from sklearn.compose import ColumnTransformer
    from sklearn.pipeline import make_pipeline

    print("Setting up the hybrid KNNImputer pipeline...")

    # 1. Initialize the KNNImputer (suitable for smaller fit samples)
    knn_imputer = KNNImputer(n_neighbors=5, weights='distance')

    # 2. Build the ColumnTransformer
    hybrid_preprocessor = ColumnTransformer([
        ('numbers', make_pipeline(knn_imputer, StandardScaler()), testing_numeric_cols),
        ('categories', OneHotEncoder(handle_unknown='ignore'), testing_categorical_cols),
    ])

    # 3. Fit on raw training features; X_train above is already transformed.
    print("Extracting 50k training sample for fitting...")
    X_train_sample = testing_train[testing_feature_cols].sample(n=min(50000, len(testing_train)), random_state=SEED)

    # 4. Fit the preprocessor ONLY on the sample
    print("Fitting preprocessor on the 50k sample...")
    hybrid_preprocessor.fit(X_train_sample)

    # 5. Preview a single batch instead of transforming all four datasets.
    X_preview = hybrid_preprocessor.transform(testing_train[testing_feature_cols].head(1000))
    print('KNN preview:', X_preview.shape)

## Three sequential comparisons — same simple neural network

Enable `RUN_COMPARISONS` at the start of Testing and run all Testing cells in order. The loading cell recreates the original training pool from the same raw source used at the top. A fixed sample of up to **1,000,000 training rows** is split 80/20 once for experiments. All cases use the same experiment training/validation rows. Original validation/test/held-out data are not used.

1. **Encoding:** PROTOCOL and L7_PROTO both one-hot versus both binary. StandardScaler and the original class distribution are fixed; other categories remain one-hot.
2. **Scaling:** selected encoding; StandardScaler versus quantile-to-normal, without resampling.
3. **Imbalance:** selected encoding/scaler; no resampling, random under/oversampling at ratios 0.25/0.5/1.0, **ADASYN**, and **ADASYN followed by Tomek Links**. Targets already reached by the natural ratio are skipped. Ratios use binary Label, not individual attack families.

Every case uses a fresh network with one 32-neuron hidden layer, Adam, **30 epochs**, the same seed and threshold 0.5. Imputation and all preprocessing are fitted on experiment training only, before resampling. Experiment validation is never resampled. Selection uses validation macro-F1, then minority average precision. The existing sample size and network budget are retained; ADASYN/Tomek nearest-neighbour searches may take substantial time at this size. Adjust `SAMPLE_ROWS` explicitly if needed.

**Paper method:** [Abdelkhalek and Mashaly, Section 3.2.3 / Algorithm 2](https://doi.org/10.1007/s11227-023-05073-x): ADASYN generates minority examples; Tomek Links then removes majority-class endpoints. `ADASYN_RATIO=1.0` requests approximate balance and `ADASYN_NEIGHBORS=5` is reduced only if the minority count requires it. The original majority label is kept fixed for cleaning even if ADASYN rounding reverses the class counts. Before/after counts and the removed-row count are displayed.

This is an adaptation to the notebook's **binary Label benchmark**. Benign examples remain present. The paper's separate attack-only multiclass experiment drops benign traffic; that is not applied to binary detection here. ADASYN interpolates in the encoded/scaled feature space, so synthetic categorical coordinates can be fractional and are used only as model inputs. It does not reconstruct raw traffic records. An impossible ADASYN case is reported as skipped; no other sampler is silently substituted. These comparisons do not change the main saved preprocessing outputs.

Implementation references: [ADASYN](https://imbalanced-learn.org/stable/references/generated/imblearn.over_sampling.ADASYN.html), [TomekLinks](https://imbalanced-learn.org/stable/references/generated/imblearn.under_sampling.TomekLinks.html).

The comparisons apply the same fixed log1p compression to the two SECOND_BYTES fields before fitting. They retain their existing legacy feature baseline. They do not evaluate the new main Encoding branch (identifier PCA, binary ports, decomposed flags and ICMP code). Main exports use the Encoding function above.

In [ ]:
if RUN_COMPARISONS:
    %pip install -q imbalanced-learn category_encoders
    from time import perf_counter
    import gc
    import numpy as np
    import pandas as pd
    from scipy import sparse
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import OneHotEncoder, StandardScaler, QuantileTransformer
    from sklearn.neural_network import MLPClassifier
    from sklearn.metrics import f1_score, average_precision_score, recall_score, confusion_matrix
    from category_encoders import BinaryEncoder
    from sklearn.experimental import enable_iterative_imputer
    from sklearn.impute import IterativeImputer
    from imblearn.under_sampling import RandomUnderSampler, TomekLinks
    from imblearn.over_sampling import RandomOverSampler, ADASYN

    SAMPLE_ROWS = 1000_000
    NN_EPOCHS = 30  # Same fixed training budget for every fresh neural network.
    RATIOS = [0.25, 0.50, 1.0]
    ADASYN_RATIO = 1.0  # Desired minority/majority ratio; actual counts may differ slightly.
    ADASYN_NEIGHBORS = 5
    # Algorithm 2 removes majority-class endpoints. Freeze the original majority label below.

    # One uniform sample shared by every trial; all eligible source classes retain natural prevalence in expectation.
    exp_sample = testing_train.sample(n=min(SAMPLE_ROWS, len(testing_train)), random_state=SEED).copy()
    exp_train, exp_val = train_test_split(exp_sample, test_size=0.20,
                                         stratify=exp_sample['Label'], random_state=SEED)
    if len(exp_sample['Label'].unique()) != 2:
        raise ValueError('The selected sample must contain both labels. Increase SAMPLE_ROWS.')
    exp_features = list(testing_feature_cols)
    exp_numeric = list(testing_numeric_cols)
    exp_categories = list(testing_categorical_cols)
    exp_protocols = [c for c in exp_features if c.upper() in ('PROTOCOL', 'L7_PROTO')]
    exp_other_categories = [c for c in exp_categories if c not in exp_protocols]
    y_exp_train = exp_train['Label'].to_numpy()
    y_exp_val = exp_val['Label'].to_numpy()
    label_counts = pd.Series(y_exp_train).value_counts()
    if len(label_counts) != 2 or not set(label_counts.index).issubset({0, 1}):
        raise ValueError('These ratio comparisons require binary Label values 0 and 1.')
    minority_label = label_counts.idxmin()
    majority_label = label_counts.idxmax()
    natural_ratio = label_counts.min() / label_counts.max()
    print(f'Sampled {len(exp_sample):,} rows; experiment train {len(exp_train):,}, validation {len(exp_val):,}')
    display(pd.crosstab(exp_sample['Attack'], exp_sample['Label']))
    print('Minority label:', minority_label, '| natural minority/majority ratio:', round(natural_ratio, 4))

In [ ]:
if RUN_COMPARISONS:
    # Same imputer and parameters as the main pipeline, fitted anew without experiment-validation leakage.
    exp_imputer = IterativeImputer(max_iter=5, random_state=SEED, skip_complete=True)
    t0 = perf_counter()
    exp_num_train = exp_imputer.fit_transform(exp_train[exp_numeric])
    exp_num_val = exp_imputer.transform(exp_val[exp_numeric])
    print(f'Shared imputation: {perf_counter() - t0:.1f}s')

    # Keep every other categorical feature unchanged across encoder/scaler experiments.
    if exp_other_categories:
        exp_other_encoder = OneHotEncoder(handle_unknown='ignore', dtype=np.float32)
        exp_cat_train = exp_other_encoder.fit_transform(exp_train[exp_other_categories])
        exp_cat_val = exp_other_encoder.transform(exp_val[exp_other_categories])
    else:
        exp_cat_train = sparse.csr_matrix((len(exp_train), 0), dtype=np.float32)
        exp_cat_val = sparse.csr_matrix((len(exp_val), 0), dtype=np.float32)

    def compare_trial(X_train_trial, X_val_trial, encoding, scaling, sampling='none', ratio=None):
        if ratio is not None and ratio <= natural_ratio:
            print(f'Skip {sampling} ratio {ratio}: natural ratio is already {natural_ratio:.3f}')
            return None
        started = perf_counter()
        adasyn_rows = None
        tomek_removed = 0
        if sampling == 'none':
            X_fit, y_fit = X_train_trial, y_exp_train
        elif sampling in ('under', 'over'):
            sampler_class = RandomUnderSampler if sampling == 'under' else RandomOverSampler
            sampler = sampler_class(sampling_strategy=ratio, random_state=SEED)
            X_fit, y_fit = sampler.fit_resample(X_train_trial, y_exp_train)
        elif sampling in ('adasyn', 'adasyn_tomek'):
            neighbours = min(ADASYN_NEIGHBORS, int(label_counts.min()) - 1)
            if neighbours < 1:
                print(f'Skip {sampling}: ADASYN requires at least two minority examples.')
                return None
            print(f'{sampling}: ADASYN ratio={ratio}, neighbours={neighbours}', flush=True)
            before_counts = pd.Series(y_exp_train).value_counts().sort_index()
            try:
                X_fit, y_fit = ADASYN(
                    sampling_strategy=ratio, n_neighbors=neighbours, random_state=SEED
                ).fit_resample(X_train_trial, y_exp_train)
            except (ValueError, RuntimeError) as error:
                # Separated neighbourhoods or very small requested additions can make ADASYN impossible.
                print(f'Skip {sampling}: {error}')
                return None
            adasyn_rows = len(y_fit)
            count_stages = {'before': before_counts,
                            'after_ADASYN': pd.Series(y_fit).value_counts().sort_index()}
            if sampling == 'adasyn_tomek':
                print('Cleaning ADASYN output with Tomek Links...', flush=True)
                X_fit, y_fit = TomekLinks(
                    sampling_strategy=[majority_label], n_jobs=-1
                ).fit_resample(X_fit, y_fit)
                tomek_removed = adasyn_rows - len(y_fit)
                count_stages['after_Tomek'] = pd.Series(y_fit).value_counts().sort_index()
            display(pd.DataFrame(count_stages).fillna(0).astype(int).rename_axis('Label'))
            print('Tomek rows removed:', tomek_removed)
        else:
            raise ValueError(f'Unknown sampler: {sampling}')
        if len(np.unique(y_fit)) != 2:
            print(f'Skip {sampling}: cleaning left fewer than two classes.')
            return None
        # Fresh, identical small neural network for each case; only input width changes with encoding.
        model = MLPClassifier(hidden_layer_sizes=(32,), activation='relu', solver='adam',
                              batch_size=min(1024, len(y_fit)), learning_rate_init=0.001,
                              alpha=1e-4, random_state=SEED)
        for epoch in range(NN_EPOCHS):
            model.partial_fit(X_fit, y_fit, classes=np.array([0, 1]))
        probability_attack = model.predict_proba(X_val_trial)[:, list(model.classes_).index(1)]
        predicted = (probability_attack >= 0.5).astype(int)
        minority_scores = probability_attack if minority_label == 1 else 1 - probability_attack
        tn, fp, fn, tp = confusion_matrix(y_exp_val, predicted, labels=[0, 1]).ravel()
        fit_counts = pd.Series(y_fit).value_counts()
        result = dict(adasyn_rows=adasyn_rows, tomek_removed=tomek_removed, encoding=encoding, scaling=scaling, sampling=sampling, target_ratio=ratio,
                      actual_ratio=fit_counts.min()/fit_counts.max(), train_rows=len(y_fit),
                      features=X_fit.shape[1], macro_F1=f1_score(y_exp_val, predicted, average='macro'),
                      AP_minority=average_precision_score(y_exp_val == minority_label, minority_scores),
                      recall_minority=recall_score(y_exp_val, predicted, pos_label=minority_label, zero_division=0),
                      recall_attack=recall_score(y_exp_val, predicted, pos_label=1, zero_division=0),
                      false_positive_rate=fp/max(1, fp+tn), fit_seconds=perf_counter()-started,
                      matrix_MB=(X_fit.data.nbytes+X_fit.indices.nbytes+X_fit.indptr.nbytes)/2**20,
                      epochs=NN_EPOCHS)
        print(encoding, scaling, sampling, ratio, '| macro F1:', round(result['macro_F1'], 4))
        return result  # Model and resampled matrix references are released after this call.

    def build_experiment_features(encoding, scaling):
        # Both protocol columns use the SAME encoding in each case; all other categories stay one-hot.
        if len(exp_protocols) != 2:
            raise ValueError('Both PROTOCOL and L7_PROTO are required for this comparison.')
        scaler = (StandardScaler() if scaling == 'standard' else
                  QuantileTransformer(output_distribution='normal', n_quantiles=min(1000, len(exp_train)),
                                      subsample=min(50_000, len(exp_train)), random_state=SEED))
        scaled_train = scaler.fit_transform(exp_num_train)
        scaled_val = scaler.transform(exp_num_val)
        for values in (scaled_train, scaled_val):
            if not np.isfinite(values).all() or np.any(np.abs(values) > np.finfo(np.float32).max):
                raise ValueError('Transformed numeric values cannot be stored safely as float32.')
        num_train = sparse.csr_matrix(scaled_train, dtype=np.float32)
        num_val = sparse.csr_matrix(scaled_val, dtype=np.float32)
        encoder = (OneHotEncoder(handle_unknown='ignore', dtype=np.float32) if encoding == 'one_hot'
                   else BinaryEncoder(cols=exp_protocols, return_df=False,
                                      handle_unknown='value', handle_missing='value'))
        proto_train = sparse.csr_matrix(encoder.fit_transform(exp_train[exp_protocols]), dtype=np.float32)
        proto_val = sparse.csr_matrix(encoder.transform(exp_val[exp_protocols]), dtype=np.float32)
        return (sparse.hstack([num_train, exp_cat_train, proto_train], format='csr'),
                sparse.hstack([num_val, exp_cat_val, proto_val], format='csr'))

    def best_case(results):
        # Same primary metric for all three comparisons; minority AP breaks ties.
        return max(results, key=lambda r: (r['macro_F1'], r['AP_minority']))

### 1. Encoding comparison

In [ ]:
if RUN_COMPARISONS:
    # 1. Encoding: BOTH protocol features one-hot versus BOTH binary. No resampling.
    encoding_results = []
    for encoding in ['one_hot', 'binary']:
        X_case_train, X_case_val = build_experiment_features(encoding, 'standard')
        encoding_results.append(compare_trial(X_case_train, X_case_val, encoding, 'standard'))
        del X_case_train, X_case_val
        gc.collect()
    display(pd.DataFrame(encoding_results).round(4))
    BEST_ENCODING = best_case(encoding_results)['encoding']
    print('Selected encoding for the scaler comparison:', BEST_ENCODING)


### 2. Scaler comparison

In [ ]:
if RUN_COMPARISONS:
    # 2. Scaling: keep the selected encoding; still no resampling.
    # Reuse its StandardScaler score because the data, model seed and settings are identical.
    scaling_results = [dict(best_case(encoding_results))]
    X_case_train, X_case_val = build_experiment_features(BEST_ENCODING, 'normal')
    scaling_results.append(compare_trial(X_case_train, X_case_val, BEST_ENCODING, 'normal'))
    del X_case_train, X_case_val
    gc.collect()
    display(pd.DataFrame(scaling_results).round(4))
    BEST_SCALER = best_case(scaling_results)['scaling']
    print('Selected encoding/scaler for imbalance testing:', BEST_ENCODING, BEST_SCALER)


### 3. Imbalance comparison — last

In [ ]:
if RUN_COMPARISONS:
    # 3. Imbalance LAST: freeze the selected encoding and scaler for every sampling case.
    imbalance_results = [dict(best_case(scaling_results))]  # Unbalanced reference, already evaluated.
    X_case_train, X_case_val = build_experiment_features(BEST_ENCODING, BEST_SCALER)
    for method in ['under', 'over']:
        for ratio in RATIOS:
            result = compare_trial(X_case_train, X_case_val, BEST_ENCODING, BEST_SCALER, method, ratio)
            if result is not None:
                imbalance_results.append(result)
    # Paper comparison: identical experiment rows, preprocessing and network budget.
    for method in ['adasyn', 'adasyn_tomek']:
        result = compare_trial(X_case_train, X_case_val, BEST_ENCODING, BEST_SCALER,
                               method, ADASYN_RATIO)
        if result is not None:
            imbalance_results.append(result)
        gc.collect()
    display(pd.DataFrame(imbalance_results).sort_values(['macro_F1', 'AP_minority'], ascending=False).round(4))
    selected_case = best_case(imbalance_results)
    print('Selected by validation macro-F1:', {k: selected_case[k] for k in
          ['encoding', 'scaling', 'sampling', 'target_ratio', 'macro_F1', 'false_positive_rate']})
    comparison_results = pd.DataFrame(
        [dict(stage='encoding', **r) for r in encoding_results] +
        [dict(stage='scaling', **r) for r in scaling_results] +
        [dict(stage='imbalance', **r) for r in imbalance_results])
    # Free experiment matrices; keep only the small result tables and selected settings.
    del X_case_train, X_case_val, exp_sample, exp_train, exp_val, exp_num_train, exp_num_val
    del exp_cat_train, exp_cat_val, exp_imputer, testing_train
    gc.collect()